##### Content of this Notebook:
- ###### SCENARIO-1: Auto-Loader
- ###### SCENARIO-2: Auto-Loader: With different schema at the source
- ###### SCENARIO-3: Slowly Changing Dimension - Intial and Incremental
- ###### SCENARIO-4: Creating Class & Functions for Windows Function
- ###### SCENARIO-5: Conditional Column
- ###### SCENARIO-6: For Loop, as per the Parameter Passed
- ###### SCENARIO-6.1: Partition of Data
- ###### SCENARIO-7: Error handling
- ###### General delta table operations

In [0]:
df = spark.read.format("csv")\
    .option("header",True)\
    .option("InferSchema",True)\
    .load("/Volumes/workspace/default/raw_data_pysparkrealtime/sales_data_first.csv")
df.display()

#### SCENARIO-1: Auto-Loader
![image_1784798008458.png](./image_1784798008458.png "image_1784798008458.png")

In [0]:
df = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.Format","csv")\
    .option("cloudFiles.schemaLocation","/Volumes/workspace/default/raw_destination/CheckPoint")\
    .load("/Volumes/workspace/default/raw_data_pysparkrealtime")

In [0]:
## .trigger(processingTime="3 seconds")\
df.writeStream.format("delta")\
    .option("checkpointLocation","/Volumes/workspace/default/raw_destination/CheckPoint")\
    .trigger(availableNow=True)\
    .start("/Volumes/workspace/default/raw_destination/data")

In [0]:
%sql
select * from delta.`/Volumes/workspace/default/raw_destination/data`

#### SCENARIO-2: Auto-Loader: With new schema a source
![image_1784799091360.png](./image_1784799091360.png "image_1784799091360.png")

In [0]:
## .option("mergeSchema",True) -> This option enables to merge the schema, and refresh the cache of Auto_loader
df.writeStream.format("delta")\
    .option("checkpointLocation","/Volumes/workspace/default/raw_destination/CheckPoint")\
    .trigger(availableNow=True)\
    .option("mergeSchema",True)\
    .start("/Volumes/workspace/default/raw_destination/data")

In [0]:
%sql
select * from delta.`/Volumes/workspace/default/raw_destination/data`
-- we got the all new count with the newly added column.

#### SCENARIO-3: Slowly Changing Dimension - Intial and Incremental

In [0]:
df = spark.read.format("csv")\
    .option("header",True)\
    .option("inferSchema",True)\
    .load("/Volumes/workspace/default/raw_csv/products_dim_table.csv")
df.display()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
df = df.select("p_id","p_name","p_category").filter(col("p_id").isNotNull())
df.display()

In [0]:
initial_run = 0

In [0]:
if (initial_run == 0):
    delta_table = DeltaTable.forPath(spark,"/Volumes/workspace/default/raw_csv_sink")
    delta_table.alias("trg").merge(df.alias("src"), "trg.p_id = src.p_id")\
                        .whenMatchedUpdateAll()\
                        .whenNotMatchedInsertAll()\
                        .execute()

else:
    df.write.format("delta")\
        .mode("append")\
        .save("/Volumes/workspace/default/raw_csv_sink")   
    df.write.saveAsTable("productDim")     

In [0]:
%sql
select * from productDim

#### SCENARIO-4: Creating Class & Functions for Windows Function:

In [0]:
df = spark.read.format("csv")\
        .option("header",True)\
        .option("inferSchema",True)\
        .load("/Volumes/workspace/default/raw_data_pysparkrealtime/sales_data_first.csv")
df.display()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
df.withColumn("dense_rank", dense_rank().over(Window.partitionBy("Month").orderBy(col("Units_Sold").desc()))).display()

In [0]:
## Step 1: Creating class called windows_function.
## Step 2: Creating a function inside it as dense_rank_func and giving the expresssion/code i want to execute inside it.

class windows_function:

    df = spark.read.format("csv")\
        .option("header",True)\
        .option("inferSchema",True)\
        .load("/Volumes/workspace/default/raw_data_pysparkrealtime/sales_data_first.csv")

    def dense_rank_func(self,new_col,part_col,order_by):
        self.df = self.df.withColumn(new_col, dense_rank().over(Window.partitionBy(part_col).orderBy(col(order_by).desc())))
        return self.df
    
    def rank_func(self,new_col,part_col,order_by):
        self.df = self.df.withColumn(new_col, rank().over(Window.partitionBy(part_col).orderBy(col(order_by).desc())))
        return self.df
    
    def row_num_func(self,new_col,part_col,order_by):
        self.df = self.df.withColumn(new_col, row_number().over(Window.partitionBy(part_col).orderBy(col(order_by).desc())))
        return self.df

In [0]:
## Call the function with the inputs and display the result
obj = windows_function()

df_dense = obj.dense_rank_func("DenseRankColumn","Month","Units_Sold")
df_dense.display()

In [0]:
obj = windows_function()
rank_df = obj.rank_func("RankColumn","Month","Units_Sold")
rank_df.display()
row_num_df = obj.row_num_func("RowNumColumn","Month","Units_Sold")
row_num_df.display()

##### Calling the function from another NB - "Class_Function_Nb"

In [0]:
%run "./Class_Function_Nb"

In [0]:
obj = validation()

df1 = obj.column_validation("Validation_Column","Units_Sold")
df1.display()

#### SCENARIO-5: Conditional Column:

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
## Writing conditional using when/otherwise (pyspark)
df.withColumn("ConditionalCol", when(col("Units_Sold")==1,"Low")\
                                .when(col("Units_Sold")==2,"Medium")\
                                .otherwise("High")
             ).display()

In [0]:
## Doing this using user defined function (python)

## Create the UDF:
def condition(x):
    if (x==1):
        return "Low"
    elif (x==2):
        return "Medium"
    else:
        return "High"

In [0]:
## Define this function as UDF
condition_udf = udf(condition)

In [0]:
## Call this UDF in the conditional column
df.withColumn("ConditionalColUDF", condition_udf("Units_Sold")).display()

## Note - It's not adviced to use python if pyspark is available to use, at it may impact on performance.

#### SCENARIO-6: For Loop, as per the Parameter Passed
##### Want to save the Data as per the applied filter which, we are saving in variable

In [0]:
var_unit_sold = [1,2,3]

In [0]:
for i in var_unit_sold:

    df = spark.read.format("csv")\
        .option("header",True)\
        .option("InferSchema",True)\
        .load("/Volumes/workspace/default/raw_data_pysparkrealtime/sales_data_first.csv")\
        .filter(col('Units_Sold') == i)

    df.write.format("csv")\
            .mode("append")\
            .option("path",f"/Volumes/workspace/default/loopdata_unitssold/UnitsSold = {i}")\
            .save()

##### SCENARIO-6.1: Partition of Data

In [0]:
df = df.withColumn("OldPartitionID", spark_partition_id())
df.display()

In [0]:
df = df.repartition(4)

In [0]:
df = df.withColumn("NewPartitionID", spark_partition_id())
df.display()

In [0]:
df.withColumn("NewPartitionID", spark_partition_id()).groupBy("NewPartitionID").count().display()

#### SCENARIO 7: Error Handling

In [0]:
## We made some changes in path of reading file, and tried handling that error, so that notebook should not stop running if any error in between

try:
    df = spark.read.format("csv")\
        .option("header",True)\
        .option("InferSchema",True)\
        .load("/Volumes/workspace/default/raw_data_pysparkrealtime/sales_data_first_WrongPath.csv")
    df.display()

except:
    print("There was an error found!")

In [0]:
## Same code but this time we want to register what was the error, using Exception
try:
    df = spark.read.format("csv")\
        .option("header",True)\
        .option("InferSchema",True)\
        .load("/Volumes/workspace/default/raw_data_pysparkrealtime/sales_data_first_WrongPath.csv")
    df.display()

except Exception as e:
    print(f"There was an error found! - {e}")

In [0]:
print("To check the continuty, but Running all")

#### General delta table operations:

In [0]:
%sql
CREATE TABLE users
(
    id INT,
    name STRING,
    email STRING
)
USING DELTA

In [0]:
%sql
INSERT INTO users
VALUES
(1, "Sushant","Sushant@gmail.com"),
(1, "Anjali", "Anjali@yahoo.com")

In [0]:
%sql
SELECT * FROM users

In [0]:
%sql
ALTER TABLE users SET TBLPROPERTIES ('delta.enableDeletionVectors' = false)

In [0]:
%sql
DESCRIBE HISTORY users

In [0]:
%sql
DESCRIBE DETAIL users

In [0]:
%sql
SELECT * FROM DELTA.`/Tables/workspace/default/users`